# Week 3 -- Function 8: Bayesian Optimisation
### Imperial College -- Bayesian Optimisation Capstone

In [ ]:
# Cell 2: Imports
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

In [ ]:
# -- WEEKLY OBSERVATIONS LOG --------------------------------------------------------------------------------
# Each week: append one new tuple  ([x1, x2, ...], y_value)  to weekly_log.
# This cell is idempotent -- safe to re-run; it never double-appends.

import numpy as np

INITIAL_SIZE = 40   # number of samples in the original course-provided data
DATA_PATH_X  = '../data/function_8/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_8/initial_outputs.npy'

weekly_log = [
    ([0.816815, 0.656459, 0.258882, 0.854757, 0.737459, 0.233756, 0.838932, 0.820863], 7.2751498504466),      # Week 1: submitted [0.816815, 0.656459, 0.258882, 0.854757, 0.737459, 0.233756, 0.838932, 0.820863], received 7.2751498504466
    ([0.856208, 0.91428, 0.313812, 0.369507, 0.710367, 0.257166, 0.646775, 0.025767], 7.627455582011601),      # Week 2: submitted [0.856208, 0.914280, 0.313812, 0.369507, 0.710367, 0.257166, 0.646775, 0.025767], received 7.627455582011601
    # Week 3: add next week's result here as ([x1, x2, ...], y_value)
]

# -- auto-sync to .npy (idempotent) ------------------------------------------------------------------
X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f"Synced {n_missing} new observation(s).")
else:
    print("Already up-to-date.")

print(f"X: {X_disk.shape} | Y: {Y_disk.shape}")
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f"Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)")


In [ ]:
# Cell 3: Load data and inspect
# Function 8: ML model with 8 hyperparameters, Maximise.
# Dimensions: learning rate, batch size, layers, dropout, regularisation,
#             activation function, optimiser type, weight range.

X = np.load('../data/function_8/initial_inputs.npy')
Y = np.load('../data/function_8/initial_outputs.npy')

#Set this once and never change it -- it's the size of the original file

print(f'Input  shape : {X.shape}   (n_samples x n_dimensions)')
print(f'Output shape : {Y.shape}  (n_samples,)')
print()

# Sort descending by Y value
X_list = list(X)
Y_list = list(Y)
pairs = sorted(zip(Y_list, X_list), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 110)
print('  All observations (sorted descending by Y)')
print('=' * 110)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join([f'{v:.5f}' for v in x_val])
    print(f'  [{i+1:2d}]  X = [{x_str}]   Y = {y_val:+.6f}{marker}')
print('=' * 110)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
x_str_best = ', '.join([f'{v:.8f}' for v in best_X])
print(f'\n  Best Y*  = {best_Y:.6f}')
print(f'  Best X*  = [{x_str_best}]')

In [ ]:
# Cell 4: Fit GP with log-transformed Y

# Log-transform to handle extreme scale differences across observations
Y_fit = np.log(np.abs(Y) + 1e-300)

# Fixed RBF kernel (course style -- no ConstantKernel, no optimisation)
kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
gp.fit(X, Y_fit)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                   : {gp.kernel_}')
print(f'  Log-marginal-likelihood  : {gp.log_marginal_likelihood_value_:.4f}')
print()

# Sanity check: predict at best known point
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
actual_log = np.log(np.abs(best_Y) + 1e-300)
x_str_best = ', '.join([f'{v:.6f}' for v in best_X])
print('  Sanity check at best known X*:')
print(f'    X*                     = [{x_str_best}]')
print(f'    GP predicted mean      = {pred_mean[0]:.4f}  (log-space)')
print(f'    Actual log(|Y*|)       = {actual_log:.4f}  (log-space)')
print(f'    GP predicted std       = {pred_std[0]:.8f}')
print('=' * 62)

In [ ]:
# Cell 5: EI (Expected Improvement) acquisition over random search (50,000 points in 8D)
# Switched from UCB -- with beta=4.0 UCB kept returning similar high-x1/x2 regions
# with only marginal gains. EI specifically searches for points likely to EXCEED
# the current best, making better use of limited queries in high-dimensional space.
# High dimensionality requires large random search to adequately cover the space.

np.random.seed(42)
X_grid = np.random.uniform(0, 1, size=(50000, 8))  # shape (50000, 8)

# GP predictions across the random grid
post_mean, post_std = gp.predict(X_grid, return_std=True)

# EI acquisition: E[max(f(x) - f*, 0)]
best_log_y  = np.log(np.abs(best_Y) + 1e-300)
improvement = post_mean - best_log_y
Z           = improvement / (post_std + 1e-9)
acquisition = improvement * norm.cdf(Z) + post_std * norm.pdf(Z)
acquisition = np.maximum(acquisition, 0)  # EI is non-negative by definition

# Next query = grid point with highest EI
best_idx = np.argmax(acquisition)
next_x   = X_grid[best_idx]
next_acq = acquisition[best_idx]

# Portal submission string
portal_string = '-'.join([f'{v:.6f}' for v in next_x])

next_x_str = ', '.join([f'{v:.6f}' for v in next_x])
print('=' * 62)
print('  EI Acquisition  (random search 50,000 pts, 8D)')
print('=' * 62)
print(f'  Grid size            : {len(X_grid)} points (random uniform, 8D)')
print(f'  Max EI value         : {next_acq:.6f}')
print(f'  Best log(|Y*|) ref   : {best_log_y:.4f}  (= log(|{best_Y:.4f}|))')
print(f'  Next query point     : [{next_x_str}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 62)

In [ ]:
# Cell 7: Summary

next_x_str = ', '.join([f'{v:.6f}' for v in next_x])
best_x_str = ', '.join([f'{v:.6f}' for v in best_X])

print('=' * 62)
print('  SUMMARY -- Bayesian Optimisation Results')
print('=' * 62)
print(f'  Function             : 8D ML Hyperparameter Tuning')
print(f'  Dims                 : lr, batch_size, layers, dropout, reg,')
print(f'                         activation_fn, optimiser_type, weight_range')
print(f'  Objective            : Maximise')
print(f'  Kernel               : RBF(length_scale=0.1, fixed)')
print(f'  Acquisition function : EI  (Expected Improvement)')
print(f'  Y transform          : log(|Y| + 1e-300)')
print(f'  Grid search          : Random uniform (50,000 points, 8D)')
print(f'  Current week         : {current_week}')
print(f'  Observations         : {len(Y)} total ({len(Y)-INITIAL_SIZE} added so far)')
print()
print(f'  Current best X*      : [{best_x_str}]')
print(f'  Current best Y*      : {best_Y:.6f}')
print()
print(f'  Next query point     : [{next_x_str}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 62)

## Week 3 -- Reflection

**Function**: F8 -- 8D ML Hyperparameter Tuning (8D)  
**Domain context**: lr, batch_size, layers, dropout, reg, activation, optimiser, weight_range

### Query
- **Method**: EI (Expected Improvement) with RBF kernel -- switching from UCB; EI will target points likely to exceed 7.627 rather than just uncertain regions.
- **Query type**: *(fill in after running notebook -- Exploration or Exploitation?)*
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after receiving result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 4
*(fill in after reflecting on result)*